In [ ]:
import pandas as pd
import numpy as np

# Define file paths (update if needed)
DATA_PATH = "data/"

# Load key datasets
men_games = pd.read_csv(DATA_PATH + "MRegularSeasonCompactResults.csv")
women_games = pd.read_csv(DATA_PATH + "WRegularSeasonCompactResults.csv")
men_tourney = pd.read_csv(DATA_PATH + "MNCAATourneyCompactResults.csv")
women_tourney = pd.read_csv(DATA_PATH + "WNCAATourneyCompactResults.csv")
men_teams = pd.read_csv(DATA_PATH + "MTeams.csv")
women_teams = pd.read_csv(DATA_PATH + "WTeams.csv")
men_seeds = pd.read_csv(DATA_PATH + "MNCAATourneySeeds.csv")
women_seeds = pd.read_csv(DATA_PATH + "WNCAATourneySeeds.csv")
men_rankings = pd.read_csv(DATA_PATH + "MMasseyOrdinals.csv")  # Only available for men

# Add a gender column to differentiate men's and women's data
men_games["Gender"] = "Men"
women_games["Gender"] = "Women"
men_tourney["Gender"] = "Men"
women_tourney["Gender"] = "Women"

# Merge regular season and tournament data
games = pd.concat([men_games, women_games, men_tourney, women_tourney], ignore_index=True)

# Create win/loss team-level stats from regular season games
def compute_team_stats(games_df):
    stats = games_df.groupby(["Season", "WTeamID"]).agg(
        Wins=("WTeamID", "count"),
        TotalPointsScored=("WScore", "sum"),
        TotalPointsAllowed=("LScore", "sum"),
    ).reset_index().rename(columns={"WTeamID": "TeamID"})

    losses = games_df.groupby(["Season", "LTeamID"]).agg(
        Losses=("LTeamID", "count"),
    ).reset_index().rename(columns={"LTeamID": "TeamID"})

    team_stats = pd.merge(stats, losses, on=["Season", "TeamID"], how="left").fillna(0)
    team_stats["WinPct"] = team_stats["Wins"] / (team_stats["Wins"] + team_stats["Losses"])
    team_stats["AvgPointsScored"] = team_stats["TotalPointsScored"] / (team_stats["Wins"] + team_stats["Losses"])
    team_stats["AvgPointsAllowed"] = team_stats["TotalPointsAllowed"] / (team_stats["Wins"] + team_stats["Losses"])
    team_stats["PointMargin"] = team_stats["AvgPointsScored"] - team_stats["AvgPointsAllowed"]

    return team_stats

# Compute team statistics
men_team_stats = compute_team_stats(men_games)
women_team_stats = compute_team_stats(women_games)

# Add gender flag for merging later
men_team_stats["Gender"] = "Men"
women_team_stats["Gender"] = "Women"

team_stats = pd.concat([men_team_stats, women_team_stats], ignore_index=True)

# Aggregate Massey Ordinals to get ranking data (only for men)
def aggregate_rankings(rank_df):
    final_rankings = rank_df[rank_df["RankingDayNum"] == 133]  # End of regular season
    rank_summary = final_rankings.groupby(["Season", "TeamID"]).agg(
        MeanRank=("OrdinalRank", "mean"),
        MedianRank=("OrdinalRank", "median"),
    ).reset_index()
    return rank_summary

men_team_ranks = aggregate_rankings(men_rankings)
men_team_ranks["Gender"] = "Men"  # Ensure correct merging

# Merge rankings with team stats (only for men)
men_team_stats = men_team_stats.merge(men_team_ranks, on=["Season", "TeamID", "Gender"], how="left")

# Merge tournament seeds (both men and women)
def process_seeds(seed_df):
    seed_df["SeedNum"] = seed_df["Seed"].str.extract('(\d+)').astype(float)  # Extract numerical seed
    return seed_df.drop(columns=["Seed"])

men_seeds = process_seeds(men_seeds)
women_seeds = process_seeds(women_seeds)
men_seeds["Gender"] = "Men"
women_seeds["Gender"] = "Women"

team_seeds = pd.concat([men_seeds, women_seeds], ignore_index=True)

# Merge team stats with seeds
team_features = team_stats.merge(team_seeds, on=["Season", "TeamID", "Gender"], how="left")

# Prepare game-level dataset
def prepare_game_data(games_df, team_features_df):
    games_df = games_df.rename(columns={"WTeamID": "Team1", "LTeamID": "Team2"})
    
    # Merge team stats for both teams
    games_df = games_df.merge(team_features_df, left_on=["Season", "Team1", "Gender"], right_on=["Season", "TeamID", "Gender"], how="left", suffixes=("_T1", ""))
    games_df = games_df.merge(team_features_df, left_on=["Season", "Team2", "Gender"], right_on=["Season", "TeamID", "Gender"], how="left", suffixes=("_T1", "_T2"))
    
    # Drop unnecessary columns
    games_df = games_df.drop(columns=["TeamID_T1", "TeamID_T2"])
    
    # Create target variable (1 if Team1 won, 0 if Team2 won)
    games_df["Result"] = 1  # Since Team1 is always the winner in original dataset
    return games_df

# Process game data
final_dataset = prepare_game_data(games, team_features)

# Save final dataset
final_dataset.to_csv("NCAA_Final_Modeling_Dataset.csv", index=False)

In [2]:
from IPython.display import display

# Display final dataset sample
display(final_dataset.head())

,Season,DayNum,Team1,WScore,Team2,LScore,WLoc,NumOT,Gender,Wins_T1,...,Wins_T2,TotalPointsScored_T2,TotalPointsAllowed_T2,Losses_T2,WinPct_T2,AvgPointsScored_T2,AvgPointsAllowed_T2,PointMargin_T2,SeedNum_T2,Result
0,1985,20,1228,81,1328,64,N,0,Men,23,...,25.0,2320.0,1871.0,5.0,0.833333,77.333333,62.366667,14.966667,1.0,1
1,1985,25,1106,77,1354,70,H,0,Men,10,...,9.0,685.0,581.0,15.0,0.375000,28.541667,24.208333,4.333333,NaN,1
2,1985,25,1112,63,1223,56,H,0,Men,18,...,17.0,1235.0,1058.0,8.0,0.680000,49.400000,42.320000,7.080000,NaN,1
3,1985,25,1165,70,1432,54,H,0,Men,12,...,11.0,756.0,638.0,12.0,0.478261,32.869565,27.739130,5.130435,NaN,1
4,1985,25,1192,86,1447,74,H,0,Men,19,...,8.0,673.0,619.0,16.0,0.333333,28.041667,25.791667,2.250000,NaN,1


In [6]:
# Let's understand the structure of our NCAA dataset better
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss, roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler

# Load the dataset
df = pd.read_csv('NCAA_Final_Modeling_Dataset.csv')

# Check the structure of the dataset
print("Dataset shape:", df.shape)

# Let's see what the Result column contains
print("\
Result column values:", df['Result'].unique())

# Let's check if there are any games from the 2025 season
print("\
Games from 2025 season:", df[df['Season'] == 2025].shape[0])

# Let's look at the most recent season data
most_recent_season = df['Season'].max()
print(f"\
Most recent season: {most_recent_season}")
print(f"Number of games in most recent season: {df[df['Season'] == most_recent_season].shape[0]}")

# Let's check if we have any sample submission files
import os
print("\
Files in directory:", os.listdir())

## Analyze theam data for 2025 season

# Let's understand what we need to predict
# Based on the submission format, we need to predict the probability of Team1 winning against Team2 for the 2025 season

# First, let's check the format of the ID column in the submission file
# The format seems to be: 2025_TeamID1_TeamID2

# Let's check the unique team IDs in the dataset
team_ids = set(df['Team1'].unique()) | set(df['Team2'].unique())
print(f"Number of unique team IDs: {len(team_ids)}")

# Let's check the distribution of teams in the 2025 season
teams_2025 = set(df[df['Season'] == 2025]['Team1'].unique()) | set(df[df['Season'] == 2025]['Team2'].unique())
print(f"Number of teams in 2025 season: {len(teams_2025)}")

# Let's check if all teams in 2025 have both men's and women's teams
men_teams_2025 = set(df[(df['Season'] == 2025) & (df['Gender'] == 'Men')]['Team1'].unique()) | set(df[(df['Season'] == 2025) & (df['Gender'] == 'Men')]['Team2'].unique())
women_teams_2025 = set(df[(df['Season'] == 2025) & (df['Gender'] == 'Women')]['Team1'].unique()) | set(df[(df['Season'] == 2025) & (df['Gender'] == 'Women')]['Team2'].unique())

print(f"Number of men's teams in 2025: {len(men_teams_2025)}")
print(f"Number of women's teams in 2025: {len(women_teams_2025)}")

# Let's check if the Result column is always 1
print(f"\
Is Result always 1? {(df['Result'] == 1).all()}")

# This suggests that Team1 is always the winner
# Let's verify this by checking if WScore > LScore
print(f"Is WScore always > LScore? {(df['WScore'] > df['LScore']).all()}")

# Let's check if there's a pattern in how Team1 and Team2 are assigned
print("\
Distribution of WLoc (Winner's location):")
print(df['WLoc'].value_counts())

Dataset shape: (331912, 28)
Result column values: [1]
Games from 2025 season: 8869
Most recent season: 2025
Number of games in most recent season: 8869
Files in directory: ['.git', '.venv', 'catboost_info', 'data', 'deepmadness.ipynb', 'exploration.ipynb', 'madbook.ipynb', 'madness.ipynb', 'madness_start.ipynb', 'march_madness_model.pkl', 'merging.ipynb', 'models', 'n2_madness.ipynb', 'n31_madness.ipynb', 'n3_madness.ipynb', 'n4_maddness.ipynb', 'n5_madness.ipynb', 'n6_madness.ipynb', 'NCAA_Final_Modeling_Dataset.csv', 'next_madness.ipynb', 'old', 'openmadness.ipynb', 'optimized_xgboost_model.pkl', 'output', 'results', 'submission.csv']
Number of unique team IDs: 749
Number of teams in 2025 season: 726
Number of men's teams in 2025: 364
Number of women's teams in 2025: 362
Is Result always 1? True
Is WScore always > LScore? True
Distribution of WLoc (Winner's location):
WLoc
H    191124
A    107922
N     32866
Name: count, dtype: int64
